# SmartStock AI — Model Evaluation

This notebook evaluates the saved Random Forest demand-forecasting model against a previous-day baseline.

**Run from the repository root.** It expects `data/train.csv` and `models/demand_model.joblib`. It loads the saved model and does not retrain it.

## 1. Load libraries and project files

In [4]:

from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import plotly.express as px
from IPython.display import display

# --------------------------------------------------
# 1. FIND PROJECT FILES
# --------------------------------------------------

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "train.csv"
MODEL_PATH = PROJECT_ROOT / "models" / "demand_model.joblib"

print("Project root:", PROJECT_ROOT)
print("Dataset exists:", DATA_PATH.exists())
print("Model exists:", MODEL_PATH.exists())

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found: {DATA_PATH}"
    )

if not MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Trained model not found: {MODEL_PATH}"
    )

# --------------------------------------------------
# 2. LOAD DATA AND TRAINED MODEL
# --------------------------------------------------

sales = pd.read_csv(DATA_PATH, parse_dates=["date"])
artifact = joblib.load(MODEL_PATH)

if isinstance(artifact, dict) and "model" in artifact:
    model = artifact["model"]

    feature_columns = artifact.get(
        "feature_columns",
        [
            "lag_1",
            "lag_7",
            "lag_14",
            "day_of_week",
            "day_of_month",
            "month",
        ],
    )

    saved_mae = artifact.get("mae")
    saved_baseline_mae = artifact.get("baseline_mae")

else:
    model = artifact

    feature_columns = [
        "lag_1",
        "lag_7",
        "lag_14",
        "day_of_week",
        "day_of_month",
        "month",
    ]

    saved_mae = None
    saved_baseline_mae = None

# --------------------------------------------------
# 3. VALIDATE DATA
# --------------------------------------------------

required_columns = {"date", "store", "item", "sales"}
missing_columns = required_columns - set(sales.columns)

if missing_columns:
    raise ValueError(
        f"Dataset is missing columns: {sorted(missing_columns)}"
    )

sales["date"] = pd.to_datetime(sales["date"])
sales = sales.sort_values(
    ["store", "item", "date"]
).reset_index(drop=True)

print("\nDATASET SUMMARY")
print("-" * 40)
print(f"Rows: {len(sales):,}")
print(f"Stores: {sales['store'].nunique()}")
print(f"Products: {sales['item'].nunique()}")
print(
    f"Date range: {sales['date'].min().date()} "
    f"to {sales['date'].max().date()}"
)
print("Model:", type(model).__name__)
print("Features:", feature_columns)

if saved_mae is not None:
    print(f"Saved model MAE: {saved_mae:.3f}")

if saved_baseline_mae is not None:
    print(f"Saved baseline MAE: {saved_baseline_mae:.3f}")

# --------------------------------------------------
# 4. PREPARE CHRONOLOGICAL TEST DATA
# --------------------------------------------------

# Use the final 30 calendar days as the test period.
test_start = sales["date"].max() - pd.Timedelta(days=29)

train_history = sales[sales["date"] < test_start].copy()
test_data = sales[sales["date"] >= test_start].copy()

# Create lag features using past observations only.
# Concatenation ensures test rows can use preceding history.
combined = pd.concat(
    [train_history, test_data],
    ignore_index=True,
)

combined = combined.sort_values(
    ["store", "item", "date"]
).reset_index(drop=True)

grouped_sales = combined.groupby(
    ["store", "item"],
    sort=False,
)["sales"]

combined["lag_1"] = grouped_sales.shift(1)
combined["lag_7"] = grouped_sales.shift(7)
combined["lag_14"] = grouped_sales.shift(14)

combined["day_of_week"] = combined["date"].dt.dayofweek
combined["day_of_month"] = combined["date"].dt.day
combined["month"] = combined["date"].dt.month

test_features = combined[
    combined["date"] >= test_start
].copy()

test_features = test_features.dropna(
    subset=feature_columns + ["sales"]
)

if test_features.empty:
    raise ValueError(
        "No test rows remain after creating lag features."
    )

X_test = test_features[feature_columns]
y_test = test_features["sales"]

# --------------------------------------------------
# 5. GENERATE MODEL PREDICTIONS
# --------------------------------------------------

predictions = model.predict(X_test)

# Previous-day baseline prediction.
baseline_predictions = test_features["lag_1"].to_numpy()

# --------------------------------------------------
# 6. EVALUATE MODEL PERFORMANCE
# --------------------------------------------------

model_mae = float(
    np.mean(np.abs(y_test.to_numpy() - predictions))
)

baseline_mae = float(
    np.mean(
        np.abs(
            y_test.to_numpy() - baseline_predictions
        )
    )
)

model_rmse = float(
    np.sqrt(
        np.mean(
            (y_test.to_numpy() - predictions) ** 2
        )
    )
)

baseline_rmse = float(
    np.sqrt(
        np.mean(
            (y_test.to_numpy() - baseline_predictions) ** 2
        )
    )
)

improvement = (
    (baseline_mae - model_mae) / baseline_mae * 100
    if baseline_mae > 0
    else 0.0
)

print("\nMODEL EVALUATION RESULTS")
print("-" * 40)
print(f"Test start date: {test_start.date()}")
print(f"Test rows: {len(test_features):,}")
print(f"Random Forest MAE: {model_mae:.3f}")
print(f"Previous-day baseline MAE: {baseline_mae:.3f}")
print(f"Random Forest RMSE: {model_rmse:.3f}")
print(f"Baseline RMSE: {baseline_rmse:.3f}")
print(f"MAE reduction vs baseline: {improvement:.2f}%")

if model_mae < baseline_mae:
    print("Result: Model outperforms the baseline.")
else:
    print("Result: Baseline performs better on this test set.")

# --------------------------------------------------
# 7. SAVE PREDICTIONS FOR ANALYSIS
# --------------------------------------------------

results = test_features[
    ["date", "store", "item", "sales"]
].copy()

results = results.rename(
    columns={"sales": "actual_sales"}
)

results["predicted_sales"] = predictions
results["baseline_prediction"] = baseline_predictions

results["absolute_error"] = np.abs(
    results["actual_sales"] - results["predicted_sales"]
)

results["baseline_absolute_error"] = np.abs(
    results["actual_sales"] - results["baseline_prediction"]
)

print("\nSAMPLE PREDICTIONS")
display(results.head(10))

# --------------------------------------------------
# 8. PLOT ACTUAL VS PREDICTED SALES
# --------------------------------------------------

daily_results = (
    results.groupby("date", as_index=False)
    .agg(
        actual_sales=("actual_sales", "sum"),
        predicted_sales=("predicted_sales", "sum"),
        baseline_prediction=("baseline_prediction", "sum"),
    )
)

fig = px.line(
    daily_results,
    x="date",
    y=[
        "actual_sales",
        "predicted_sales",
        "baseline_prediction",
    ],
    title="Actual vs Predicted Daily Sales",
    labels={
        "date": "Date",
        "value": "Total sales",
        "variable": "Series",
    },
)

fig.show()

# --------------------------------------------------
# 9. PLOT DAILY ABSOLUTE ERROR
# --------------------------------------------------

daily_errors = (
    results.groupby("date", as_index=False)
    .agg(
        model_mae=("absolute_error", "mean"),
        baseline_mae=("baseline_absolute_error", "mean"),
    )
)

error_fig = px.line(
    daily_errors,
    x="date",
    y=["model_mae", "baseline_mae"],
    title="Daily Mean Absolute Error: Model vs Baseline",
    labels={
        "date": "Date",
        "value": "Mean absolute error",
        "variable": "Method",
    },
)

error_fig.show()

# --------------------------------------------------
# 10. SAVE EVALUATION RESULTS
# --------------------------------------------------

output_path = PROJECT_ROOT / "notebooks" / "evaluation_results.csv"

results.to_csv(output_path, index=False)

print("\nEvaluation complete.")
print(f"Prediction results saved to: {output_path}")


Project root: c:\Users\honey\smartstock-ai
Dataset exists: True
Model exists: True

DATASET SUMMARY
----------------------------------------
Rows: 913,000
Stores: 10
Products: 50
Date range: 2013-01-01 to 2017-12-31
Model: RandomForestRegressor
Features: ['store', 'item', 'lag_1', 'lag_7', 'lag_14', 'day_of_week', 'day_of_month', 'month']
Saved model MAE: 6.318
Saved baseline MAE: 9.219

MODEL EVALUATION RESULTS
----------------------------------------
Test start date: 2017-12-02
Test rows: 15,000
Random Forest MAE: 6.318
Previous-day baseline MAE: 9.219
Random Forest RMSE: 8.200
Baseline RMSE: 12.346
MAE reduction vs baseline: 31.47%
Result: Model outperforms the baseline.

SAMPLE PREDICTIONS


,date,store,item,actual_sales,predicted_sales,baseline_prediction,absolute_error,baseline_absolute_error
1796,2017-12-02,1,1,16,21.679192,19.0,5.679192,3.0
1797,2017-12-03,1,1,31,18.426248,16.0,12.573752,15.0
1798,2017-12-04,1,1,7,19.288385,31.0,12.288385,24.0
1799,2017-12-05,1,1,20,15.898895,7.0,4.101105,13.0
1800,2017-12-06,1,1,17,21.596702,20.0,4.596702,3.0
1801,2017-12-07,1,1,18,19.334810,17.0,1.334810,1.0
1802,2017-12-08,1,1,17,19.920372,18.0,2.920372,1.0
1803,2017-12-09,1,1,27,17.857053,17.0,9.142947,10.0
1804,2017-12-10,1,1,20,27.917766,27.0,7.917766,7.0
1805,2017-12-11,1,1,20,11.522692,20.0,8.477308,0.0


ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

## 2. Create time-based features

Lag features use only earlier sales for the same store and product. The final 30 calendar days are used as the chronological test period.

In [ ]:
group_cols = ["store", "item"]
grouped_sales = sales.groupby(group_cols, sort=False)["sales"]
sales["lag_1"] = grouped_sales.shift(1)
sales["lag_7"] = grouped_sales.shift(7)
sales["lag_14"] = grouped_sales.shift(14)
sales["day_of_week"] = sales["date"].dt.dayofweek
sales["day_of_month"] = sales["date"].dt.day
sales["month"] = sales["date"].dt.month

evaluation_data = sales.dropna(subset=feature_columns + ["sales"]).copy()
last_date = sales["date"].max()
test_start = last_date - pd.Timedelta(days=29)
test_data = evaluation_data[evaluation_data["date"] >= test_start].copy()
if test_data.empty:
    raise ValueError("The chronological test set is empty. Check the dataset dates and feature creation.")

print(f"Test period: {test_data['date'].min().date()} to {test_data['date'].max().date()}")
print(f"Test rows: {len(test_data):,}")
print(f"Test dates: {test_data['date'].nunique()}")

## 3. Compare Random Forest with the previous-day baseline

MAE (Mean Absolute Error) is the average absolute difference between actual and predicted sales. Lower is better. The baseline predicts each store–product's sales using its previous day's sales (`lag_1`).

In [ ]:
X_test = test_data[feature_columns]
test_data["model_prediction"] = model.predict(X_test)
test_data["baseline_prediction"] = test_data["lag_1"]
test_data["model_absolute_error"] = (test_data["sales"] - test_data["model_prediction"]).abs()
test_data["baseline_absolute_error"] = (test_data["sales"] - test_data["baseline_prediction"]).abs()

model_mae = test_data["model_absolute_error"].mean()
baseline_mae = test_data["baseline_absolute_error"].mean()
improvement_pct = (baseline_mae - model_mae) / baseline_mae * 100 if baseline_mae else np.nan
metrics = pd.DataFrame({"Model": ["Random Forest", "Previous-day baseline"], "MAE (sales units)": [model_mae, baseline_mae]})
display(metrics.round(3))
print(f"MAE reduction vs baseline: {improvement_pct:.2f}%")
if saved_mae is not None:
    print(f"MAE stored in model artifact: {saved_mae:.3f}")
if saved_baseline_mae is not None:
    print(f"Baseline MAE stored in model artifact: {saved_baseline_mae:.3f}")
print("Result: Random Forest has lower MAE than baseline." if model_mae < baseline_mae else "Result: Random Forest did not outperform the baseline on this test period.")

## 4. Plot daily actual versus predicted sales

This chart aggregates sales across all store–product combinations for each test date.

In [ ]:
daily_comparison = test_data.groupby("date", as_index=False).agg(
    actual_sales=("sales", "sum"),
    random_forest_prediction=("model_prediction", "sum"),
    previous_day_baseline=("baseline_prediction", "sum"),
)
daily_long = daily_comparison.melt(id_vars="date", var_name="series", value_name="sales_units")
fig = px.line(daily_long, x="date", y="sales_units", color="series",
              title="Daily Actual Sales vs Predictions — Final 30 Days",
              labels={"date":"Date", "sales_units":"Total sales units", "series":"Series"})
fig.update_layout(legend_title_text="")
fig.show()

## 5. Review errors by store

This table helps identify whether the model's errors vary across stores.

In [ ]:
store_errors = test_data.groupby("store", as_index=False).agg(
    model_mae=("model_absolute_error", "mean"),
    baseline_mae=("baseline_absolute_error", "mean"),
    test_rows=("sales", "size"),
)
store_errors["mae_reduction_pct"] = (
    (store_errors["baseline_mae"] - store_errors["model_mae"])
    / store_errors["baseline_mae"].replace(0, np.nan) * 100
)
display(store_errors.round(3).sort_values("model_mae"))

## 6. Inspect the largest prediction errors

Use these rows to investigate unusual demand spikes or limitations in the current feature set.

In [ ]:
error_review = test_data[["date", "store", "item", "sales", "model_prediction", "baseline_prediction", "model_absolute_error"]].copy()
display(error_review.sort_values("model_absolute_error", ascending=False).head(15).round(2))

## 7. Conclusions and limitations

After running the notebook, summarize the results shown above.

- **MAE** is measured in sales units; it is not a percentage accuracy score.
- Compare both approaches on exactly the same test rows.
- Keep time-series validation chronological; do not randomly shuffle dates.
- Results depend on the selected test window and feature assumptions, and may not guarantee future performance.
- **Next experiment:** compare another forecasting model on the same test dates and report MAE (and optionally RMSE).